# Experiment 2 · "If I train on artificial damage only, will it detect real damage?"

**Question:** the deployment-critical one — how well does a model trained **only
on hand-made (artificial, `KA`) damage + healthy** bearings generalize to
**real fatigue damage** (`KI`, `KB` combination)?

**Protocol:** identical to Experiment 1 except the training set is reduced to
`partition == train` (KA + healthy). All **6 independent models** are trained on
that set and evaluated **once** on the real/combination bearings (healthy reused
as the shared reference). Every model's result is shown — no cherry-picking.
A per-model decision threshold is tuned on a KA-only out-of-fold CV (never on the
real test data).

In [ ]:
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'pyproject.toml').exists():
        sys.path.insert(0, str(_p)); break
from src import experiments as ex
import numpy as np, pandas as pd

df, X = ex.load_data()
cache = ex.load_cache()

tr_mask  = (df.partition == "train").to_numpy()
tt_mask  = ((df.partition == "test") | (df.label == "healthy")).to_numpy()
tr, X_tr = df[tr_mask].reset_index(drop=True), X[tr_mask]
tt, X_tt = df[tt_mask].reset_index(drop=True), X[tt_mask]

print("train (artificial + healthy):", tr.bearing.nunique(), "bearings")
print("test  (real/comb + healthy) :", tt.bearing.nunique(), "bearings")

### All-model transfer results (run once per model, nothing hidden)

In [ ]:
tab, verdicts, probs = ex.transfer_full(tr, X_tr, tt, X_tt, cache)

In [ ]:
display(tab.sort_values("bearing_auroc", ascending=False))

### Verdict performance on real damage at a per-model tuned threshold (≤10 % healthy FPR)

In [ ]:
display(verdicts[["model", "threshold", "precision", "recall", "specificity", "f1", "accuracy"]])

### The cleanest transfer number — real-only bearings (KI, excluding KB's artificial component)

`KB` bearings carry *additional* artificial damage, so they partially share the
training signature. **KI-only** isolates the pure artificial → real-fatigue gap.

In [ ]:
from utils.metrics import auroc, aupr, bearing_level, bearing_scores

ki_mask = (((tt.label == "damaged") & (tt.bearing.str.startswith("KI")))
           | (tt.label == "healthy")).to_numpy()
rows = []
for name, p_cal in probs:
    sub, p_sub = tt[ki_mask], p_cal[ki_mask]
    y = (sub.label == "damaged").to_numpy().astype(int)
    bl = bearing_level(sub, p_sub)
    rows.append({"model": name, "window_auroc": auroc(y, p_sub), **bearing_scores(bl)})
display(pd.DataFrame(rows).sort_values("bearing_auroc", ascending=False))

### Conclusion *updated after execution*